# Figure della tesi

Notebook di lavoro per le immagini che finiscono in `docs/tesi/`.
Ogni figura sta in un modulo `.py` accanto a questo file: qui si tara e si
guarda, li' sta il codice. Cosi' la tesi non dipende dall'esecuzione di un
notebook e le figure si rigenerano da riga di comando
(`python notebooks/bok_fig2.py`).

| Figura | Modulo | Output |
|---|---|---|
| Pannello 3D stile Bok et al. (2018), Fig. 2 | `bok_fig2.py` | `docs/tesi/figures/fig_panel_3d.{pdf,png}` |

---

## 1. Pannello 3D — replica della Figure 2 di Bok et al. (2018)

**Cos'e' l'originale** (letto sul PDF a 200 dpi, p. 627 dell'articolo):
una **mesh 3D unica** — una faccetta per cella (mese x serie), reticolo
sottile sugli spigoli, **nessun riempimento** fino a una base. Le punte
nascono dal fatto che serie adiacenti hanno valori scorrelati. Sotto, su un
piano staccato, la heatmap della **stessa matrice**.

**Le serie sono standardizzate**: la didascalia dice *"displays the
standardized time series"* e l'asse verticale si chiama *"Standard deviation
from mean"*. Quindi z-score per colonna, sui dati gia' trasformati
(MoM_log, QoQ_log_ar, diff_ppt, level) di `dataset_final.csv`.

### Priorita': dato &rarr; geometria &rarr; leggibilita' &rarr; estetica

- **niente trimming** alla prima data comune;
- **niente riempimento** dei buchi: no ffill, no zeri, no interpolazione;
- i NaN diventano **buchi nella mesh** e **celle non colorate** nella heatmap;
- heatmap e superfici condividono matrice, `meshgrid`, extent e orientamento:
  la heatmap non ha ne' dati ne' indici propri, quindi non puo' risultare
  tagliata o disallineata.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

import bok_fig2 as B

tab = B.diagnostica()

### I controlli di fedelta' al dato

La cella sopra stampa il riepilogo; `tab` ha il dettaglio per serie.
Quello che va guardato prima di fidarsi della figura:

- **37 serie, 499 mesi, 1985-01 &rarr; 2026-07, 16.9% di NaN**;
- **20 serie su 37 non partono dall'inizio** (PPIFIS dal 2009, PCEC96 dal
  2007, Empire e JOLTS dal 2001, ISM_NMI dal 1997, un blocco dal 1992-93);
- **8 serie hanno gap interni**: cinque ne hanno 2 (buchi veri di un mese),
  le tre trimestrali ne hanno 328 ciascuna perche' sono osservate un mese
  su tre.

Tutto questo deve **vedersi** nella figura. Le fasce bianche sul pavimento
e i bordi irregolari a sinistra sono esattamente la *ragged edge* di cui
parla la sezione 1.2 dell'introduzione.

In [ ]:
tab.sort_values("copertura_%").head(12)

### Perche' le superfici NON devono essere nastri paralleli

Due cause, tutte e due nel rendering, nessuna nei dati.

**1. L'ordine delle serie.** Ordinarle per categoria produce otto blocchi
contigui, cioe' otto bande separate. Qui l'ordine e' quello del **dataset**:
lo stesso colore ricompare a quote diverse e le superfici si intersecano.
Le categorie servono solo al colore, non alla posizione.

**2. Il rapporto fra scala verticale e passo fra serie.** E' il parametro
che decide tutto:

$$\frac{\text{altezza di 1 sd}}{\text{passo fra due serie}}
=\frac{\texttt{aspect}_z/\text{ampiezza zlim}}{\texttt{aspect}_y/37}$$

Sotto ~1 ogni serie resta confinata nella propria fascia orizzontale e la
figura degenera in 37 nastri. Sopra ~2 le superfici si compenetrano e si
ottiene la "montagna" dell'originale. `figura_bok2` stampa il rapporto a
ogni chiamata: **e' l'unico numero da guardare** quando la geometria non
convince.

In [ ]:
fig, ax = B.figura_bok2(
    start="1985-01-01",
    zclip=5.0,            # troncamento SOLO grafico, va dichiarato
    interp_display=False,  # nessun riempimento dei buchi
    box_aspect=(2.05, 1.30, 1.00),
    zoom=1.26,
    elev=25,
    azim=-62,
)
B.salva(fig, "fig_panel_3d")

### Le manopole

| Parametro | Cosa fa |
|---|---|
| `box_aspect=(x, y, z)` | **il parametro decisivo** — vedi il rapporto sopra |
| `gap_pavimento` | distanza fra la mesh e il piano della heatmap, in multipli di `zclip` |
| `zoom`, `elev`, `azim` | inquadratura e camera |
| `zclip` | troncamento: ad aprile 2020 alcune serie del lavoro toccano z &asymp; **-20**; senza troncare, tutto il resto viene schiacciato sullo zero |
| `alpha` | trasparenza delle faccette, per vedere le superfici dietro |
| `interp_display` | **lasciare False.** Se True interpola *solo dentro lo span gia' osservato* (mai all'indietro): chiuderebbe il pettine delle tre trimestrali ma e' un riempimento, e va dichiarato |
| `rasterize` | rasterizza mesh e reticolo nel PDF: 499x37 faccette in vettoriale pesano decine di MB |

In [ ]:
# Taratura a vista. Non salva.
for ba, el in [((2.05, 1.30, 1.00), 25), ((2.05, 1.10, 1.25), 22), ((2.20, 1.50, 0.85), 28)]:
    B.figura_bok2(box_aspect=ba, elev=el)
    print(f"  box_aspect={ba}  elev={el}\n")

### Nota sui colori

La palette e' quella di riferimento **desaturata al 18%** verso un neutro
caldo. Non e' quella di Bok et al., ed e' una scelta: i loro colori
(mattone, verde bosco, arancio, blu notte, verde chiaro, azzurro, rosa
polvere, grigio) passati al validatore **falliscono** —
*Housing and construction* contro *Retail and consumption* sta a
**&Delta;E 2.5 in deuteranopia**, cioe' un lettore daltonico rosso-verde
non distingue le due categorie; grigio e rosa cadono sotto la soglia di
croma e leggono come "nessun colore".

Il 18% e' il punto di equilibrio trovato per tentativi col validatore:
&mdash; al 12% e al 18% passa tutto, al 25% la coppia verde/arancio scende
nella banda di rischio, al 32% il rosa cade sotto la soglia di croma.

In [ ]:
for cat, hexv in B.PALETTE.items():
    print(f"{hexv}  {cat:26s} {len(B.CATEGORIES[cat])} serie")